# 82514 · Sesión S14 — Taller: el PID de una articulación, con gravedad, saturación y anti-windup

**Bloque 5** · jueves 5 de noviembre de 2026 · 2 h  ·  IQS Universitat Ramon Llull

**Qué hace este cuaderno.** El hombro del cuaderno S13a, ahora en un plano vertical y con gravedad, mandado en tensión con un techo de ±24 V. Se sintoniza un PID discreto, término a término, contra una especificación (Mp < 5 % y ts < 1 s del colgado a la horizontal, libre y con el brazo sujeto el primer segundo), se ve el windup que provoca el techo, se aplican sus dos remedios y se mide la sensibilidad. La sección 5 es el entregable.

**Se apoya en:** Lynch y Park (2017), cap. 11; Corke (2023), cap. 9; De Silva et al. (2016), cap. 4; documentación de python-control.

**Cómo usarlo en clase.** Sigue el guion de la sesión S14 en las diapositivas del bloque 5. Ejecuta la celda de instalación una sola vez al empezar; en Colab tarda unos segundos. Las celdas marcadas **Ejercicio** son para que los trabajen los estudiantes: las soluciones están al final del cuaderno.

---

In [ ]:
# Ejecutar una sola vez. En Colab tarda unos segundos.
import importlib, subprocess, sys

def asegurar(mods):
    faltan = []
    for pip_name, import_name in mods:
        try:
            importlib.import_module(import_name)
        except ImportError:
            faltan.append(pip_name)
    if faltan:
        print('Instalando:', ' '.join(faltan))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + faltan, check=False)
    else:
        print('Todo instalado ya.')

asegurar([('numpy', 'numpy'), ('scipy', 'scipy'), ('matplotlib', 'matplotlib'), ('control==0.10.2', 'control')])

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import control as ct
from scipy.optimize import brentq

np.set_printoptions(precision=4, suppress=True)
plt.rcParams['figure.figsize'] = (9, 3.4)
plt.rcParams['axes.grid'] = True
IQS_AZUL, IQS_VERDE = '#1B2A80', '#1FA355'
GRIS, AMBAR, NAVY, ROJO = '#8A8FA8', '#B9770E', '#2E3FA3', '#C0392B'
COMA = FuncFormatter(lambda v, p: f'{v:g}'.replace('.', ','))   # ejes con coma decimal

def ejes(*axs):
    for ax in axs:
        ax.xaxis.set_major_formatter(COMA); ax.yaxis.set_major_formatter(COMA)

def coma(x, fmt='.3g'):
    """Un número con coma decimal, para los resultados y los rótulos de las figuras."""
    return format(x, fmt).replace('.', ',')

print('python-control', ct.__version__, '- listo.')
if ct.__version__ != '0.10.2':
    print('AVISO: este cuaderno se probó con python-control 0.10.2. Ejecuta  !pip install -q control==0.10.2  y reinicia el entorno.')

> **Aviso para Colab.** La primera celda instala python-control 0.10.2, la versión de todo el bloque 5, en unos segundos. Este cuaderno no depende de los de S13: define sus propios datos.
>
> **En clase, por parejas:** ronda 1, secciones 1 a 3 (la 1 se ejecuta sin detenerse); ronda 2, secciones 4 y 5. Los ejercicios 1 a 3, en casa: las soluciones están al final.
>
> **Se sube:** al acabar la ronda 2, este cuaderno ejecutado, con la tabla de la sección 5 y vuestras tres líneas de interpretación, a la Tarea «Ganancias del PID y métricas» de Moodle: Archivo → Descargar → Descargar .ipynb; una entrega por pareja. Admite entregas hasta las 23:59 del jueves 5 de noviembre.

## 1. El hombro en vertical y el simulador

*En clase (ronda 1), ejecuta las celdas de esta sección sin detenerte: definen la planta y el simulador, y sus cifras son las de las diapositivas 25 a 27. En casa, léela entera.*

**La planta.** El hombro del cuaderno S13a: el 2R de B4 (a1 = 0,35 m, a2 = 0,2 m) con 1 kg en el codo y 1 kg en la mano, el motor de S7 (24 V, R = 2 Ω, K = 0,05, J_m = 10⁻⁴ kg·m²) mandado en tensión y la reductora de 50. Ahora el brazo está en un **plano vertical**: la gravedad tira de él. El codo, estirado y quieto. q se mide desde la horizontal, como en el 2R de B4: colgado, q = −90°; horizontal, q = 0.

En tensión, es la ecuación de S13 con una entrada más, la gravedad:

`q̈ + 4,630·q̇ = 1,852·(V − 7,06·cos q)`

Colgado, cos q = 0 y la gravedad no pide nada; en la horizontal, 7,06 V: los 8,83 N·m del peso, pasados a voltios. Multiplicada por M = 0,675, es la misma ecuación en pares, la forma de los libros:

`M·q̈ + b·q̇ + τ_g(q) = τ`,  es decir,  `0,675·q̈ + 3,125·q̇ + 8,83·cos q = 1,25·V`

- M = G²·J' = 0,425 + 0,25 = 0,675 kg·m²: la inercia que ve el hombro, la del brazo (J_L = 0,425) más la del rotor (G²·J_m = 0,25).
- b = G²·K²/R = 3,125 N·m·s/rad: la contratensión, hecha amortiguador.
- τ = (G·K/R)·V = 1,25·V, el par de entrada, en N·m (30 N·m con 24 V); τ_g = 8,83·cos q, el de la gravedad. τ_m = 0,216 s sigue siendo un tiempo.

**El ensayo.** Del brazo colgado (−90°) a la horizontal (0°), estirado, con gravedad y con el techo de ±24 V: un salto de 90°, 1,571 rad. Dos veces: **libre**, y **sujeto el primer segundo**, como por un obstáculo.

**La especificación.** En los dos ensayos, M_p < 5 % y t_s < 1 s. M_p es lo que se pasa de la horizontal, en % del salto; t_s, el tiempo que tarda en entrar en la banda del 2 % del salto (1,8°) y no volver a salir; en el ensayo sujeto, contado desde que se suelta. Una **terna** son los tres valores K_p, K_i y K_d.

In [ ]:
V_MAX, R, L, K, JM, G = 24.0, 2.0, 1e-3, 0.05, 1e-4, 50     # el motor de S7 y la reductora (S13a)
A1, A2, M1, M2, GRAV = 0.35, 0.2, 1.0, 1.0, 9.81           # el 2R de B4 con sus masas
H = 1e-3                                                    # el paso del controlador: 1 ms
Q0, QREF = -np.pi/2, 0.0                                    # el ensayo: del colgado a la horizontal
KTAU = G*K/R                                                # 1,25 N·m de par por voltio (con el rotor parado)
BEQ = G**2*K**2/R                                           # la b de la forma en par [N·m·s/rad]: la B_eq = K²/R de S13a, vista desde la articulación

def postura(q2=0.0, m_mano=M2):
    """El hombro con el codo fijo en q2: q'' = -a·q' + bv·(V - vg·cos q)."""
    JL = M1*A1**2 + m_mano*(A1**2 + A2**2 + 2*A1*A2*np.cos(q2))
    Jp = JM + JL/G**2
    tm = Jp*R/K**2                                          # tau_m [s]
    g0 = GRAV*(M1*A1 + m_mano*(A1 + A2*np.cos(q2)))         # gravedad en la horizontal [N·m]
    return dict(a=1/tm, bv=1/(G*K*tm), vg=g0/KTAU, tm=tm, M=G**2*Jp, g=g0)   # bv: rad/s² por voltio

ESTIRADO, PLEGADO, MANO_2KG = postura(0.0), postura(np.pi), postura(0.0, m_mano=2.0)
a, bv = ESTIRADO['a'], ESTIRADO['bv']                       # los números de la ecuación en tensión
print(f"M = {coma(ESTIRADO['M'], '.3f')} kg·m²   b = {coma(BEQ, '.3f')} N·m·s/rad   {coma(KTAU, '.2f')} N·m por voltio: {24*KTAU:.0f} N·m a 24 V")
print(f"sostener el brazo en la horizontal: {coma(ESTIRADO['g'], '.2f')} N·m = {coma(ESTIRADO['vg'], '.2f')} V")
print(f"en el código: a = 1/τm = {coma(a, '.3f')} 1/s, bv = {coma(bv, '.3f')} rad/s² por voltio (= 1,25/M = {coma(KTAU/ESTIRADO['M'], '.3f')})")
for nombre, p in (('plegado', PLEGADO), ('mano de 2 kg', MANO_2KG)):
    print(f"{nombre:12s}: τm = {coma(p['tm'], '.4f')} s, la gravedad pide {coma(p['vg'], '.2f')} V en la horizontal")

En el código, `a` = 1/τ_m = 4,630 y `bv` = 1,852 (rad/s² por voltio): son los números de la ecuación en tensión de la diapositiva 26. La b de la forma en par (3,125 N·m·s/rad, la contratensión hecha amortiguador) es otra cosa: `bv` = 1,25/M. Las otras dos posturas del taller, para medir la sensibilidad en la sección 5: el brazo **plegado** (el codo a 180°: menos inercia y la mitad de gravedad) y el brazo estirado con **2 kg en la mano**.

**El controlador: el bucle de la diapositiva 27.** Un PID discreto que calcula la tensión una vez por milisegundo (h = 1 ms; en el código, `H`):

- `e = q_ref - q`: el error.
- `uD = -Kd*(q - q_ant)/h`: el término D, un amortiguador virtual. La derivada es una diferencia: la q de ahora menos la de antes, entre h. Va sobre la salida medida y no sobre el error, porque el salto de la referencia sería un golpe: con K_d = 1,82, 1,571 rad en un milisegundo pedirían 1,82·1,571/0,001 = 2860 V en una muestra.
- `V = Kp*e + uI + uD`: la ley PID. K_p·e es un muelle virtual (K_p en V/rad).
- `uI = uI + Ki*e*h`: el término I, una memoria. La integral es una suma (K_i en V/(rad·s)); u_I, en voltios.
- `Vs = min(max(V, -24), 24)`: el techo del motor.
- `q_ant = q`: para la vuelta siguiente.

El orden importa: la tensión se calcula con el término I acumulado hasta la vuelta anterior, y se integra después (en la sección 4 se verá por qué). Por 1,25, las ganancias pasan a N·m.

`simula()` ejecuta ese bucle. Entre dos cálculos la tensión se mantiene, y el brazo avanza un milisegundo con un paso de Runge-Kutta de orden 4 (RK4): la integración numérica que en S13a hacía `solve_ivp`, aquí escrita a mano con paso fijo; evalúa la aceleración cuatro veces por paso y las promedia. Con el argumento `remedio`, el bucle cambia en una línea (sección 4); con `comp`, suma una compensación de la gravedad (sección 2); con `sujeto`, retiene el brazo ese tiempo. `metricas()` mide M_p, t_s y el resto; `informe()` los imprime en una línea; `dibuja()` pinta q(t) con la banda y la vertical de 1 s (y, si se pide, la tensión y el término I); `barrido_ki()` prueba varios K_i con K_p y K_d fijos.

In [ ]:
def simula(Kp, Ki, Kd, remedio=None, i_max=None, v_max=V_MAX, p=ESTIRADO, comp=0.0, sujeto=0.0, T=4.0,
           q0=Q0, q_ref=QREF):
    """El ensayo del taller con el PID discreto de la diapositiva 27 (h = 1 ms, D sobre la salida).
    remedio: None, 'cond' (no integrar mientras satura) o 'lim' (|uI| <= i_max, en V).
    v_max=None: sin techo. comp: voltios de la compensación de gravedad (V += comp·cos q).
    sujeto: segundos con el brazo sujeto en q0. p: la postura (ESTIRADO, PLEGADO, MANO_2KG u otra).
    Devuelve t, q, la V aplicada, la V pedida y el término I, uI."""
    a, bv, vg = p['a'], p['bv'], p['vg']
    techo = np.inf if v_max is None else v_max
    n = int(round(T/H))
    t = np.arange(n + 1)*H
    q = np.empty(n + 1); Va = np.zeros(n + 1); Vp = np.zeros(n + 1); UI = np.zeros(n + 1)
    q[0] = q0; w = 0.0; q_ant = q0; uI = 0.0
    acel = lambda q_, w_, V_: -a*w_ + bv*(V_ - vg*np.cos(q_))    # q'' de la planta
    for k in range(n):
        # ---- el bucle de la diapositiva 27, una vez por milisegundo ----
        e = q_ref - q[k]                            # el error
        uD = -Kd*(q[k] - q_ant)/H                   # derivada: una diferencia
        V = Kp*e + uI + uD + comp*np.cos(q[k])      # la ley PID (+ la compensación, si la hay)
        if remedio != 'cond' or abs(V) < techo:     # integración condicional: no sumar mientras satura
            uI = uI + Ki*e*H                        # la integral: una suma
        if remedio == 'lim':
            uI = min(max(uI, -i_max), i_max)        # límite del término I
        Vs = min(max(V, -techo), techo)             # el techo del motor
        q_ant = q[k]                                # para la vuelta siguiente
        Va[k], Vp[k], UI[k] = Vs, V, uI
        # ---- la planta: un milisegundo con Vs fija (RK4) ----
        if t[k] < sujeto - 1e-12:                   # sujeto: el brazo no se mueve
            q[k + 1] = q[k]; w = 0.0
            continue
        k1q, k1w = w, acel(q[k], w, Vs)
        k2q, k2w = w + H/2*k1w, acel(q[k] + H/2*k1q, w + H/2*k1w, Vs)
        k3q, k3w = w + H/2*k2w, acel(q[k] + H/2*k2q, w + H/2*k2w, Vs)
        k4q, k4w = w + H*k3w, acel(q[k] + H*k3q, w + H*k3w, Vs)
        q[k + 1] = q[k] + H/6*(k1q + 2*k2q + 2*k3q + k4q)
        w = w + H/6*(k1w + 2*k2w + 2*k3w + k4w)
    Va[n], Vp[n], UI[n] = Va[n-1], Vp[n-1], UI[n-1]
    return dict(t=t, q=q, V=Va, Vpedida=Vp, uI=UI, v_max=v_max, sujeto=sujeto, q0=q0, q_ref=q_ref, vg=vg)

def metricas(r):
    """Mp (% del salto), ts (banda del 2 % del salto, desde que se suelta), error final [°],
    V pedida máxima, tiempo saturado y término I máximo; y si cumple Mp < 5 % y ts < 1 s."""
    t, q = r['t'], r['q']
    suelto = t >= r['sujeto'] - 1e-12
    tt, qq = t[suelto] - r['sujeto'], q[suelto]
    salto = r['q_ref'] - r['q0']
    banda = 0.02*abs(salto)
    Mp = max(0.0, (np.sign(salto)*(qq - r['q_ref'])).max()/abs(salto)*100)
    fuera = np.nonzero(np.abs(qq - r['q_ref']) > banda)[0]       # muestras fuera de la banda
    if len(fuera) == 0:
        ts = 0.0
    elif fuera[-1] + 1 < len(tt) and np.abs(qq[-200:] - r['q_ref']).max() <= banda:
        ts = tt[fuera[-1] + 1]                                   # entra en la banda y ya no sale
    else:
        ts = np.nan                                              # no entra, o no se queda (los 0,2 s finales)
    tsat = 0.0 if r['v_max'] is None else (np.abs(r['Vpedida']) > r['v_max']).sum()*H
    return dict(Mp=Mp, ts=ts, ef=np.degrees(r['q_ref'] - q[-1]), Vmax=np.abs(r['Vpedida']).max(), tsat=tsat,
                uImax=np.abs(r['uI']).max(), cumple=(Mp < 5 and not np.isnan(ts) and ts < 1.0))

def informe(r, nombre):
    """Una línea: Mp, ts, error final, V pedida máxima, tiempo saturado y término I máximo."""
    m = metricas(r)
    ts = 'no entra' if np.isnan(m['ts']) else coma(m['ts'], '.2f') + ' s'
    print(f"{nombre:38s} Mp {coma(m['Mp'], '5.1f')} %  ts {ts:>8}  error final {coma(m['ef'], '6.2f')}°"
          f"  V pedida máx {coma(m['Vmax'], '5.1f')} V  saturado {coma(m['tsat'], '.2f')} s"
          f"  término I máx {coma(m['uImax'], '5.2f')} V" + ('  cumple' if m['cumple'] else ''))

def dibuja(*rs, etiquetas=None, tension=False, titulo='Del colgado a la horizontal'):
    """q(t) en grados, con la banda del 2 % del salto y la vertical de 1 s (desde que se suelta).
    Con tension=True, también la tensión (aplicada; a trazos, la pedida) y el término I."""
    colores = (IQS_AZUL, ROJO, IQS_VERDE, AMBAR, GRIS, NAVY)
    etiquetas = etiquetas or [None]*len(rs)
    fig, axs = plt.subplots(1, 3 if tension else 1, figsize=(14, 3.6) if tension else (9, 3.6), squeeze=False)
    axs = axs[0]
    r0 = rs[0]
    qr, banda = np.degrees(r0['q_ref']), 0.02*abs(np.degrees(r0['q_ref'] - r0['q0']))
    axs[0].axhspan(qr - banda, qr + banda, color=IQS_VERDE, alpha=0.15)
    for r, et, col in zip(rs, etiquetas, colores*3):
        axs[0].plot(r['t'], np.degrees(r['q']), color=col, lw=2, label=et)
        if r['sujeto'] > 0:
            axs[0].axvspan(0, r['sujeto'], color=GRIS, alpha=0.08)
        axs[0].axvline(r['sujeto'] + 1.0, color=col if r['sujeto'] > 0 else '#999', lw=1, ls=':')
        if tension:
            axs[1].plot(r['t'], r['Vpedida'], color=col, lw=1, ls='--')
            axs[1].plot(r['t'], r['V'], color=col, lw=2)
            if r['v_max'] is not None:
                axs[1].axhline(r['v_max'], color=ROJO, lw=1, ls=':'); axs[1].axhline(-r['v_max'], color=ROJO, lw=1, ls=':')
            axs[2].plot(r['t'], r['uI'], color=col, lw=2)
    axs[0].set_xlabel('t [s]'); axs[0].set_ylabel('q [°]'); axs[0].set_title(titulo, fontsize=10)
    if any(etiquetas):
        axs[0].legend(fontsize=8.5, loc='lower right')
    if tension:
        axs[1].set_xlabel('t [s]'); axs[1].set_ylabel('V [V]'); axs[1].set_title('La tensión (a trazos, la pedida)', fontsize=10)
        axs[2].axhline(r0['vg'], color=IQS_VERDE, lw=1.2, ls='--')
        axs[2].set_xlabel('t [s]'); axs[2].set_ylabel('término I [V]')
        axs[2].set_title(f"El término I (a trazos, lo que pide la gravedad: {coma(r0['vg'], '.3g')} V)", fontsize=10)
    ejes(*axs); plt.tight_layout(); plt.show()

def barrido_ki(Kp, Kd, kis, **kw):
    """Mp y ts con Kp y Kd fijos, para cada Ki de kis. Con v_max=None, el ensayo libre sin techo (ronda 1).
    Con el techo (el de serie, 24 V), el libre y el sujeto 1 s, y «cumple» solo si cumplen los dos (ronda 2)."""
    kw.setdefault('T', 6.0)
    p = kw.get('p', ESTIRADO)
    sin_techo = 'v_max' in kw and kw['v_max'] is None
    rem = {None: 'sin remedio', 'cond': 'integración condicional',
           'lim': f"límite del término I a ±{coma(kw.get('i_max') or 0, 'g')} V"}[kw.get('remedio')]
    print(f"Kp = {coma(Kp, '.2f')} V/rad · Kd = {coma(Kd, '.2f')} V·s/rad · cota de Ki: {coma((p['a'] + p['bv']*Kd)*Kp, '.0f')} · "
          + ('sin techo, ensayo libre' if sin_techo else f"techo ±{coma(kw.get('v_max', V_MAX), 'g')} V, {rem}, libre y sujeto 1 s"))
    def celda(m):
        ts = 'no entra' if np.isnan(m['ts']) else coma(m['ts'], '.2f')
        return f"{coma(m['Mp'], '7.1f')} {ts:>9}"
    if sin_techo:
        print(f"{'Ki':>6} {'Mp [%]':>7} {'ts [s]':>9}")
    else:
        print(f"{'':6} {'libre':>17} {'sujeto 1 s':>17}")
        print(f"{'Ki':>6} {'Mp [%]':>7} {'ts [s]':>9} {'Mp [%]':>7} {'ts [s]':>9}")
    for Ki in kis:
        ml = metricas(simula(Kp, Ki, Kd, **kw))
        if sin_techo:
            print(f"{coma(Ki, '6.1f')} {celda(ml)}" + ('  <- cumple' if ml['cumple'] else ''))
        else:
            ms = metricas(simula(Kp, Ki, Kd, sujeto=1.0, **kw))
            print(f"{coma(Ki, '6.1f')} {celda(ml)} {celda(ms)}" + ('  <- cumple los dos' if ml['cumple'] and ms['cumple'] else ''))

# Comprobaciones: sin tensión, el brazo colgado no se mueve; con los 7,06·cos q de la gravedad, se queda en la horizontal
r = simula(0, 0, 0, v_max=None, T=2.0)
print(f"V = 0 desde el colgado: q(2 s) = {coma(np.degrees(r['q'][-1]), '.3f')}°")
r = simula(0, 0, 0, v_max=None, T=3.0, q0=0.0, comp=ESTIRADO['vg'])
print(f"V = 7,06·cos q desde la horizontal: q(3 s) = {coma(np.degrees(r['q'][-1]), '.3f')}°")

**Para la entrega 2 (lunes 16 de noviembre): vuestra articulación.** El simulador sirve para cualquier articulación que se escriba en la misma forma, en tensión: q̈ = −a·q̇ + bv·(V − vg·cos q), con a = 1/τ_m y bv = ganancia/τ_m, donde la ganancia es la de la planta en rad/s por voltio (0,4 en el hombro) y vg, la tensión que pide la gravedad en la horizontal. En la celda siguiente, cambiad τ_m, la ganancia, la gravedad y el techo de vuestra articulación, y usad `simula(..., p=MIA, v_max=VMAX_MIA)`: el resto funciona igual. Con los datos del hombro, que son los que trae, da lo mismo que `ESTIRADO`. La rueda de un robot móvil, en llano, no tiene gravedad en su giro: VG_MIA = 0. La tabla de la sección 5, en cambio, mide siempre el hombro (estirado, plegado y con 2 kg en la mano, a 24 V): para vuestra articulación, llamad a `simula` y a `barrido_ki` con `p=MIA, v_max=VMAX_MIA` (y `q0`, `q_ref` si vuestro ensayo es otro).

In [ ]:
# Para la entrega 2: vuestra articulación, en la misma forma (en tensión)
TM_MIA, K0_MIA, VG_MIA, VMAX_MIA = 0.216, 0.4, 7.06, 24.0   # tau_m [s], ganancia [rad/s por voltio], gravedad [V], techo [V]
MIA = dict(a=1/TM_MIA, bv=K0_MIA/TM_MIA, vg=VG_MIA)            # simula(..., p=MIA, v_max=VMAX_MIA)
informe(simula(40, 30, 5, remedio='lim', i_max=VG_MIA, p=MIA, v_max=VMAX_MIA), f"MIA: 40/30/5, límite de {coma(VG_MIA, 'g')} V")

## 2. P, PD y compensación

*En clase, ronda 1. Lo que sale está en las diapositivas 28 a 30.*

**El P con gravedad.** Parado, el P da K_p·e y la gravedad pide 7,06·cos q: el brazo se para donde las dos se igualan. Con la referencia en 0, q = −e, y el reposo cumple K_p·e = 7,06·cos e. `brentq`, de scipy, busca la raíz de esa ecuación en un intervalo donde la función cambia de signo. La cuenta lineal, e = 7,06/K_p, supone cos e = 1: exagera el error cuando es grande. En S13, sin gravedad, parado, la tensión y el error eran cero; ahora sostener el brazo pide 7,06 V, y un P solo los da con error. Subir K_p reduce el error, pero por encima de 24/1,571 = 15,3 V/rad el P arranca pidiendo más de 24 V.

In [ ]:
rp6 = simula(6.08, 0, 0, T=6.0)      # 6,08 × 1,571 = 9,6 V al arrancar: no satura
rp13 = simula(13.5, 0, 0, T=6.0)     # 13,5 × 1,571 = 21,2 V: tampoco
informe(rp6, 'P 6,08 V/rad, 24 V'); informe(rp13, 'P 13,5 V/rad, 24 V')
for Kp in (6.08, 13.5):
    e = brentq(lambda e: Kp*e - ESTIRADO['vg']*np.cos(e), 0, np.pi/2)   # parado: Kp·e = 7,06·cos e
    print(f"Kp = {coma(Kp, '5.2f')}: se para a {coma(np.degrees(e), '.1f')}° de la horizontal (la cuenta lineal, 7,06/Kp: {coma(np.degrees(ESTIRADO['vg']/Kp), '.1f')}°)")
print(f"el P arranca con más de 24 V si Kp > 24/1,571 = {coma(24/(QREF - Q0), '.1f')} V/rad")
dibuja(rp6, rp13, etiquetas=['P: ' + r'$K_p$' + ' = 6,08 V/rad', 'P: ' + r'$K_p$' + ' = 13,5 V/rad'], titulo='El P con gravedad: se para antes')

**El PD: colocar los polos.** Sin gravedad y con V = K_p·e − K_d·q̇, el polinomio característico del lazo es s² + (4,630 + 1,852·K_d)·s + 1,852·K_p. Comparando con s² + 2·ζ·ω_n·s + ω_n²: ω_n² = 1,852·K_p y 2·ζ·ω_n = 4,630 + 1,852·K_d. K_p coloca ω_n; K_d, ζ; y la parte real, −ζ·ω_n, ya no está fija en −2,315 como con el P de S13. Despejando, las fórmulas de la diapositiva 29:

`K_p = ω_n²/1,852` y `K_d = (2·ζ·ω_n − 4,630)/1,852`; en el código, `Kp = wn**2/bv` y `Kd = (2*zeta*wn - a)/bv`.

En par (por 1,25), son las de los libros: 1,25·K_p = M·ω_n² y 1,25·K_d = 2·ζ·ω_n·M − b. Con ζ = 0,8 y ω_n = 5 rad/s, polos en −4 ± 3j.

En python-control, con el D sobre la salida, V = K_p·(q_ref − q − (K_d/K_p)·q̇): el camino directo es K_p·P(s) y la realimentación, 1 + (K_d/K_p)·s, que es `ct.tf([Kd/Kp, 1], [1])`. Sin gravedad, `step_info` mide la respuesta.

In [ ]:
zeta, wn = 0.8, 5.0
Kp = wn**2/bv; Kd = (2*zeta*wn - a)/bv           # las fórmulas de la diapositiva 29
print(f"Kp = {coma(Kp, '.2f')} V/rad   Kd = {coma(Kd, '.2f')} V·s/rad")
print(f"en par: M·wn² = {coma(ESTIRADO['M']*wn**2, '.3f')} N·m/rad (= 1,25·Kp)   2·zeta·wn·M − b = {coma(2*zeta*wn*ESTIRADO['M'] - BEQ, '.3f')} N·m·s/rad (= 1,25·Kd)")
P = ct.tf([bv], [1, a, 0])                        # Q(s)/V(s), sin gravedad
pd = ct.feedback(Kp*P, ct.tf([Kd/Kp, 1], [1]))    # el D sobre la salida: realimenta q + (Kd/Kp)·q'
print('polos del lazo con el PD:', '  '.join(coma(x, '.5g') for x in np.round(ct.poles(pd), 3)))
si = ct.step_info(pd, T=np.linspace(0, 6, 60001))
print(f"sin gravedad: Mp = {coma(si['Overshoot'], '.2f')} %   ts = {coma(si['SettlingTime'], '.3f')} s")

**Con gravedad, el D no sostiene.** Parado, q̇ = 0 y el D no actúa: el PD deja el error de su P, 26,8° con 13,5 V/rad. Un remedio es sumar 7,06·cos q, calculado con el modelo y la q medida, no con el error: la **compensación de gravedad**. Parado, K_p·e = 0. En `simula`, son los voltios del argumento `comp`. Su precio, el modelo: con 2 kg en la mano la gravedad pide 11,38 V y el modelo sigue dando 7,06; plegado, compensa de más. El I no necesita modelo: sección 3.

In [ ]:
rpd = simula(Kp, 0, Kd, T=6.0)                                        # el PD de −4 ± 3j, con gravedad
rcomp = simula(Kp, 0, Kd, comp=ESTIRADO['vg'], T=6.0)                 # + 7,06·cos q: la compensación
rcomp2 = simula(Kp, 0, Kd, comp=ESTIRADO['vg'], p=MANO_2KG, T=6.0)    # la misma compensación, con 2 kg en la mano
informe(rpd, 'PD 13,5/1,82, 24 V')
informe(rcomp, 'PD + 7,06·cos q')
informe(rcomp2, 'PD + 7,06·cos q, mano de 2 kg')
dibuja(rpd, rcomp, rcomp2, etiquetas=['PD', 'PD + compensación', 'PD + compensación, 2 kg en la mano'],
       titulo='El PD con gravedad, y con la gravedad compensada')

## 3. El PID, sin techo

*En clase, ronda 1, con el ejercicio de clase. Lo que sale está en la diapositiva 31.*

**El término I, una memoria.** u_I = K_i·∫e dt crece mientras queda error, hasta dar los 7,06 V que sostienen el brazo; entonces, parado, e = 0. Pero sube un orden: sin gravedad, el polinomio del lazo es de grado 3,

`s³ + (4,630 + 1,852·K_d)·s² + 1,852·K_p·s + 1,852·K_i = s³ + a₂·s² + a₁·s + a₀`

(a₂, a₁ y a₀ son sus coeficientes; no confundirlos con la `a` del código, 1/τ_m).

**La cota de K_i.** Con los tres coeficientes positivos, un polinomio de grado 3 es estable si y solo si a₂·a₁ > a₀. Es la cuenta de la ganancia última de S13 (diapositiva 18): en el borde hay un par de polos sobre el eje imaginario, s = ±j·ω; sustituyendo s = j·ω, la parte imaginaria da ω² = a₁, y la real, a₂·ω² = a₀; en el borde, a₂·a₁ = a₀. De ahí la cota:

`K_i < (4,630 + 1,852·K_d)·K_p`; en par, con las ganancias en N·m, `1,25·K_i < (b + 1,25·K_d)·1,25·K_p/M`.

La estrategia: primero K_p y K_d, que colocan los polos; luego, K_i, por debajo de su cota. La celda siguiente la comprueba con `ct.poles` al 99 % y al 101 % de la cota. El lazo con el PID, sin gravedad, se escribe directamente con su numerador, 1,852·(K_p·s + K_i), y el polinomio de arriba.

In [ ]:
for Kp_, Kd_ in ((13.5, 1.82), (40.0, 5.0)):
    cota = (a + bv*Kd_)*Kp_
    print(f"Kp = {coma(Kp_, 'g')}, Kd = {coma(Kd_, 'g')}: cota de Ki = ({coma(a, '.3f')} + {coma(bv, '.3f')}·{coma(Kd_, 'g')})·{coma(Kp_, 'g')} = {coma(cota, '.1f')}")
    for Ki_ in (0.99*cota, 1.01*cota):
        lazo = ct.tf([bv*Kp_, bv*Ki_], [1, a + bv*Kd_, bv*Kp_, bv*Ki_])
        polos = ct.poles(lazo)
        print(f"   Ki = {coma(Ki_, '6.1f')}: polos {'  '.join(coma(x, '.5g') for x in np.round(polos, 3))} -> {'estable' if polos.real.max() < 0 else 'inestable'}")

**El I sobre el PD de 13,5 y 1,82**, sin techo: la subida es lenta y el I se carga en ella, así que tarda o se pasa.

In [ ]:
ri10 = simula(13.5, 10, 1.82, v_max=None, T=6.0)
ri20 = simula(13.5, 20, 1.82, v_max=None, T=6.0)
informe(ri10, 'PID 13,5/10/1,82, sin techo')
informe(ri20, 'PID 13,5/20/1,82, sin techo')
dibuja(ri10, ri20, etiquetas=[r'$K_i$' + ' = 10', r'$K_i$' + ' = 20'], titulo='El I sobre el PD de 13,5 y 1,82, sin techo')

### Ejercicio de clase (ronda 1; su terna es la de la ronda 2)

Elegid ζ entre 0,75 y 0,85 y ω_n entre 8 y 10 rad/s. Calculad K_p y K_d con las fórmulas de la diapositiva 29 (con ζ = 0,81 y ω_n = 8,6 salen 40 y 5). Después, buscad un K_i por debajo de su cota que cumpla M_p < 5 % y t_s < 1 s sin techo, y anotad la terna: es la de la ronda 2. `barrido_ki()` se ejecuta tal cual: imprime M_p y t_s para cada K_i.

In [ ]:
zeta, wn = 0.81, 8.6                        # vuestra elección: zeta de 0,75 a 0,85; wn de 8 a 10
Kp = wn**2/bv; Kd = (2*zeta*wn - a)/bv      # las fórmulas de la diapositiva 29
barrido_ki(Kp, Kd, np.arange(10, 40.1, 2.5), v_max=None)   # se ejecuta tal cual: Mp y ts para cada Ki
# Anotad la terna que cumpla (Kp, Ki, Kd): es la de la ronda 2, en la sección 4.

## 4. El techo, el windup y un remedio

*En clase, ronda 2. Lo que sale está en las diapositivas 33 a 35.*

**El techo de 24 V.** Al arrancar, el error es el salto, 1,571 rad, y el P pide K_p·1,571: con 40 V/rad, 62,8 V. El motor da 24. Saturado, el lazo está abierto: más error no da más tensión. `simula` aplica el techo de serie, `v_max=24.0`; en la sección 3 se quitaba con `v_max=None`.

**El windup.** Mientras el motor satura, el error sigue y la integral también: al llegar a la referencia, el término I da más de lo que pide la gravedad y empuja de más. Es el **windup**: en inglés, dar cuerda, como a un reloj. En el ensayo sujeto, algo retiene el brazo el primer segundo: el I suma K_i·1,571 rad durante ese segundo y, al soltarlo, empuja con todo.

Escribid vuestra terna de la ronda 1 en la primera línea (si no llegasteis, dejad la de ejemplo, 40, 30 y 5) y ejecutad: el ensayo libre y el sujeto, a 24 V y sin remedio, con la tensión y el término I.

In [ ]:
TERNA_R1 = (40.0, 30.0, 5.0)        # vuestra terna de la ronda 1: Kp [V/rad], Ki [V/(rad·s)], Kd [V·s/rad]
Kp, Ki, Kd = TERNA_R1
libre = simula(Kp, Ki, Kd, T=6.0)                   # a 24 V, sin remedio
sujeto = simula(Kp, Ki, Kd, sujeto=1.0, T=6.0)      # retenido el primer segundo
informe(simula(Kp, Ki, Kd, v_max=None, T=6.0), 'libre, sin techo')
informe(libre, 'libre, 24 V, sin remedio')
informe(sujeto, 'sujeto 1 s, 24 V, sin remedio')
print(f"término I al soltar (t = 1 s): {coma(sujeto['uI'][int(round(1.0/H))], '.1f')} V; la gravedad pide {coma(ESTIRADO['vg'], '.2f')} V")
dibuja(libre, sujeto, etiquetas=['libre', 'sujeto 1 s'], tension=True, titulo='A 24 V y sin remedio')

Más del 100 % es pasar por encima de la vertical (con 40/45/5 sujeto, 117,6 %): en un robot real, el tope mecánico, el final de carrera físico de la articulación.

**Dos remedios, y en los dos se toca una línea del bucle:**

- **Integración condicional:** no sumar mientras satura, `if abs(V) < 24: uI = uI + Ki*e*h`. No necesita saber nada del brazo, pero el I empieza a cargar tarde. En `simula`, `remedio='cond'`. Por eso el bucle calcula V antes de integrar: la condición se decide con la tensión de esta vuelta.
- **Límite del término I:** `uI = min(max(uI, -7.06), 7.06)`, que no pase de lo que pide la gravedad. Usa el modelo, como la compensación. En `simula`, `remedio='lim', i_max=7.06`.

La celda siguiente es la tabla de la diapositiva 35, con K_p = 40 y K_d = 5 a 24 V, y la figura con las cuatro curvas libres.

In [ ]:
casos = (('sin remedio', 30, {}), ('condicional', 30, dict(remedio='cond')),
         ('condicional', 45, dict(remedio='cond')), ('límite de 7,06 V', 30, dict(remedio='lim', i_max=7.06)))
libres = []
for nombre, Ki_, kw in casos:
    rl = simula(40, Ki_, 5, T=6.0, **kw)
    informe(rl, f'{nombre}, Ki = {Ki_}, libre')
    informe(simula(40, Ki_, 5, sujeto=1.0, T=6.0, **kw), f'{nombre}, Ki = {Ki_}, sujeto 1 s')
    libres.append(rl)
dibuja(*libres, etiquetas=[f'{n}, ' + r'$K_i$' + f' = {k}' for n, k, _ in casos],
       titulo=r'$K_p$' + ' = 40 y ' + r'$K_d$' + ' = 5, a 24 V: los dos remedios')

### Ejercicio de clase (ronda 2)

Elegid un remedio y retocad las ganancias hasta cumplir en los dos ensayos. `barrido_ki()` mide, para cada K_i, el ensayo libre y el sujeto a 24 V, y marca los K_i que cumplen los dos. Con la condicional, K_i tiene que subir: el I solo carga cuando deja de saturar. Si ningún K_i cumple, retocad K_p o K_d (con las fórmulas de la sección 3) y volved a barrer.

El límite de 7,06 V es lo que pide la gravedad del modelo: si la carga cambia, ya no basta (ejercicio 3).

In [ ]:
# Ejercicio de clase (ronda 2): vuestra terna, con un remedio, en los dos ensayos
Kp, Ki, Kd = TERNA_R1
barrido_ki(Kp, Kd, np.arange(10, 100.1, 5), remedio='cond')                  # integración condicional
# barrido_ki(Kp, Kd, np.arange(10, 100.1, 5), remedio='lim', i_max=7.06)    # o el límite del término I

## 5. La tabla del entregable

*En clase: es lo que se sube.*

Cambiad solo las cuatro líneas de arriba de la celda: la pareja, la terna, el remedio y, con el límite, su valor. La celda imprime el ensayo, la especificación, la terna con su cota de K_i y el remedio, y seis filas: los dos ensayos sin remedio y con él, con «¿cumple?», y la misma terna plegado y con 2 kg en la mano, que se miden y no tienen que cumplir. Debajo, una figura con los cuatro casos estirados.

Para subirlo: Archivo → Descargar → Descargar .ipynb, y a la Tarea «Ganancias del PID y métricas» de Moodle, una entrega por pareja, antes de la puesta en común.

In [ ]:
# ── El entregable: cambiad solo estas cuatro líneas ──
PAREJA  = 'Apellido1 y Apellido2'
TERNA   = (40.0, 30.0, 5.0)        # Kp [V/rad], Ki [V/(rad·s)], Kd [V·s/rad]
REMEDIO = 'lim'                    # 'cond' (integración condicional) o 'lim' (límite del término I)
I_MAX   = 7.06                     # solo con 'lim': el límite del término I, en voltios

# ── A partir de aquí, nada que cambiar ──
assert REMEDIO in ('cond', 'lim'), "REMEDIO tiene que ser 'cond' o 'lim'"
Kp, Ki, Kd = TERNA
rem = dict(remedio=REMEDIO, i_max=I_MAX if REMEDIO == 'lim' else None)
nombre_rem = 'integración condicional' if REMEDIO == 'cond' else f'límite del término I a ±{I_MAX:g} V'
print(f'ENTREGABLE S14 · {PAREJA}')
print('Ensayo: del colgado (−90°) a la horizontal (0°), brazo estirado, con gravedad; techo ±24 V;')
print('        PID discreto (h = 1 ms, D sobre la salida); 6 s de simulación.')
print('Especificación: Mp < 5 % y ts < 1 s, banda del 2 % del salto (1,8°), libre y sujeto 1 s (ts desde que se suelta).')
print(f'Terna: Kp = {Kp:g} V/rad, Ki = {Ki:g} V/(rad·s), Kd = {Kd:g} V·s/rad · cota de Ki: {(a + bv*Kd)*Kp:.0f} · remedio: {nombre_rem}'.replace('.', ','))
print(f"{'caso':40s} {'Mp [%]':>7} {'ts [s]':>8} {'error final [°]':>16} {'V pedida máx [V]':>17}  ¿cumple?")
casos = (('libre, sin remedio', {}, ESTIRADO, 0.0, False), ('sujeto 1 s, sin remedio', {}, ESTIRADO, 1.0, False),
         ('libre, con el remedio', rem, ESTIRADO, 0.0, False), ('sujeto 1 s, con el remedio', rem, ESTIRADO, 1.0, False),
         ('plegado, con el remedio (se mide)', rem, PLEGADO, 0.0, True),
         ('mano de 2 kg, con el remedio (se mide)', rem, MANO_2KG, 0.0, True))
res = {}
for caso, kw, p, suj, mide in casos:
    res[caso] = r = simula(Kp, Ki, Kd, p=p, sujeto=suj, T=6.0, **kw)
    m = metricas(r)
    ts = 'no entra' if np.isnan(m['ts']) else f"{m['ts']:.2f}"
    cumple = '—' if mide else ('sí' if m['cumple'] else 'no')
    print(f"{caso:40s} {m['Mp']:7.1f} {ts:>8} {m['ef']:16.2f} {m['Vmax']:17.1f}  {cumple}".replace('.', ','))
dos = metricas(res['libre, con el remedio'])['cumple'] and metricas(res['sujeto 1 s, con el remedio'])['cumple']
print('Con el remedio, cumple en los dos ensayos:', 'sí' if dos else 'no')
dibuja(*(res[c] for c, *_ in casos[:4]), etiquetas=[c for c, *_ in casos[:4]], titulo=f'Entregable S14 · {PAREJA}')

### Vuestra interpretación (tres líneas)

*Haced doble clic en esta celda y sustituid cada punto por una línea vuestra, con las cifras de la tabla.*

1. Dónde cae vuestra terna en la región de ternas que cumplen (el barrido de la sección 4 os da la franja de K_i) y qué habéis sacrificado.
2. Qué hace vuestro remedio en vuestra respuesta.
3. Qué pasa plegado y con 2 kg, y qué haríais.

## 6. En casa: tres ejercicios

*En casa, sin entrega: las soluciones están al final.*

### Ejercicio 1 (en casa): el PI

Quitad el D (K_d = 0). ¿Hay algún PI que cumpla M_p < 5 % y t_s < 1 s sin techo? Mirad el coeficiente de s² del polinomio de la sección 3 con K_d = 0. Pista: en s³ + a₂·s² + a₁·s + a₀, a₂ es la suma de los tres polos cambiada de signo.

In [ ]:
# Ejercicio 1: el PI (Kd = 0), sin techo
# for Kp in (4, 10, 20):
#     barrido_ki(Kp, 0.0, (5, 7, 10, 15, 20), v_max=None)

### Ejercicio 2 (en casa): el amplificador de corriente

Con el amplificador de corriente de S7, el motor se manda en corriente: el par es K·i y la contratensión ya no frena. La planta queda q̈ = 3,704·(i − 3,532·cos q), sin término en q̇: 3,704 = K·G/M, en rad/s² por amperio, y 3,532 A sostienen el brazo en la horizontal. Con `p=CORRIENTE` y `v_max=None`, `simula` manda corriente, en amperios: donde dice V, leed i. Probad un P de 1 A/rad desde el colgado; después, el PD de ζ = 0,8 y ω_n = 5 con las fórmulas de la sección 2 (aquí, a = 0 y bv = 3,704), sin compensación y con ella (3,532·cos q, en amperios).

In [ ]:
# Ejercicio 2: el mando en corriente (S7)
CORRIENTE = dict(a=0.0, bv=K*G/ESTIRADO['M'], vg=ESTIRADO['g']/(G*K))   # q'' = 3,704·(i − 3,532·cos q)
print(f"bv = {coma(CORRIENTE['bv'], '.3f')} rad/s² por amperio   vg = {coma(CORRIENTE['vg'], '.3f')} A")
# r = simula(1.0, 0, 0, p=CORRIENTE, v_max=None, T=6.0)          # un P de 1 A/rad
# informe(r, 'P 1 A/rad'); dibuja(r)
# Kp_i = ...; Kd_i = ...                                          # el PD de zeta = 0,8 y wn = 5
# informe(simula(Kp_i, 0, Kd_i, p=CORRIENTE, v_max=None, T=6.0), 'PD en corriente')
# informe(simula(Kp_i, 0, Kd_i, p=CORRIENTE, v_max=None, T=6.0, comp=CORRIENTE['vg']), 'PD + compensación')

### Ejercicio 3 (en casa): el límite y la carga

Con 40/30/5 y el límite del término I, ¿qué límite sostiene la mano de 2 kg, y qué hace ese límite con la de 1 kg? ¿Hay algún límite que sirva para las dos cargas? Comparadlo con la integración condicional y K_i = 45.

In [ ]:
# Ejercicio 3: el límite y la carga
# for i_max in (7.06, 8.5, 11.38):
#     for nombre, p in (('1 kg', ESTIRADO), ('2 kg', MANO_2KG)):
#         informe(simula(40, 30, 5, remedio='lim', i_max=i_max, p=p, T=6.0), f'límite {i_max} V, mano de {nombre}')

---

## Soluciones

**Ejercicio de clase de la sección 3 (ronda 1).** Con ζ de 0,75 a 0,85 y ω_n de 8 a 10 rad/s, K_p sale de 34,6 a 54,0 V/rad y K_d de 3,98 a 6,68 V·s/rad. Con ζ = 0,81 y ω_n = 8,6, K_p = 39,9 y K_d = 5,02, que se redondean a 40 y 5 (la cota de K_i, 556). Sin techo cumplen K_i de 20 a 30, y con la terna redondeada, 40/30/5 da 3,9 % y 0,97 s, y 40/20/5, 0 % y 0,48 s. En toda la caja, K_i = 25 cumple sin techo. Por ejemplo, con ζ = 0,8 y ω_n = 8 (K_p = 34,6 y K_d = 4,41), K_i de 20 a 25; con ζ = 0,75 y ω_n = 10 (54,0 y 5,60), de 22,5 a 35.

**Ejercicio de clase de la sección 4 (ronda 2).** Sin remedio no cumple los dos ensayos ninguna terna, con ningún K_d: el ensayo sujeto lo rompe todo (40/30/5 sujeto: 78,9 % y 4,65 s, con el término I a 47 V al soltar). Con K_d = 5, cumplen los dos ensayos 88 ternas con la condicional (con K_p = 40, K_i de 35 a 50: 40/45/5 da 1,6 % y 0,53 s) y 280 con el límite de 7,06 V (K_p de 22,5 a 60, K_i desde 15 o 20: 40/30/5 da 1,4 % y 0,49 s, libre y sujeto). Con otros ζ y ω_n de la ronda 1, con la condicional K_i sube (con ζ = 0,8 y ω_n = 8, de 30 a 42,5); con el límite, basta K_i desde 15 o 17,5.

**Ejercicio 1.** Ninguno: de 2000 PI (K_p de 2 a 80, K_i de 1 a 150), no cumple ninguno; el mejor con M_p < 5 % tarda 2,03 s. PI 10/10: 1,6 % y 2,41 s; 20/20: 25,7 %. Sin D, el coeficiente de s² es 4,630, haga lo que haga el controlador: los tres polos suman −4,63, y el más lento no puede estar más a la izquierda de −4,63/3 = −1,54. Hace falta el D, que es el que mueve la parte real.

**Ejercicio 2.** El P de 1 A/rad oscila sin amortiguarse entre −90,1° y −48,8°: sin contratensión no hay término en q̇, y los polos quedan sobre el eje imaginario (±1,92j con 1 A/rad, sin gravedad). El PD de ζ = 0,8 y ω_n = 5 pide K_p = 25/3,704 = 6,75 A/rad y K_d = 8/3,704 = 2,16 A·s/rad, y da exactamente la misma respuesta que en tensión: 26,8° de error con gravedad; compensado, 1,5 % y 0,75 s. Con los mismos polos, la misma respuesta; sin contratensión, todo el amortiguamiento lo pone el D.

**Ejercicio 3.** Con 2 kg en la mano, el límite de 7,06 V deja 6,1° de error; el de 8,5 V, 4,1°; el de 11,38 V (lo que pide la mano de 2 kg) la sostiene (5,6 % y 0,89 s), pero con 1 kg ya no actúa, porque el término I no pasa de 11 V: vuelve el 7,2 % y 1,93 s de sin remedio. Por debajo de 11,38 V, la mano de 2 kg queda con error; por encima, el límite no frena el windup con 1 kg: con esta terna, ningún límite sirve para las dos cargas. La condicional con K_i = 45 sostiene las dos: 1,6 % y 0,53 s con 1 kg; 0 % y 1,73 s con 2 kg. El límite gana si la carga es conocida; la condicional, si cambia.

---

## Para llevarse de esta sesión

**Cada término, su trabajo.** El P es un muelle y, con gravedad, deja error; el D es un amortiguador que coloca ζ y mueve la parte real; el I quita el error, dentro de su cota, y se ajusta después de K_p y K_d.

**La compensación de gravedad usa el modelo:** se calcula con el modelo y la q medida, no con el error. Funciona con la carga del modelo; con otra, deja error.

**Con techo, el integrador se dispara:** el windup. Se remedia en una línea del bucle: no integrar mientras satura, o limitar el término I a lo que pide la gravedad.

**Una terna cumple en un ensayo,** con su banda, su salto, su techo y su carga. En otra postura o con otra carga, se vuelve a medir: es la sensibilidad.

En S15 (lunes 9 de noviembre), el espacio de estados, la dinámica del brazo entero y el par calculado.

*Cuaderno del curso 82514 Mecatrónica y Robótica · IQS Universitat Ramon Llull · curso 2026/27*

*© Guillermo Reyes Carmenaty · Publicado bajo [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es): puedes usarlo, adaptarlo y redistribuirlo, incluso con fines comerciales, siempre que reconozcas la autoría.*